In [0]:
df_silver_icd10 = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_icd10_codes")

In [0]:
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

# 1. Define window specification ordered by the business key for stable SK generation
window_spec = Window.orderBy("icd10_code")

# 2. Generate Surrogate Key: AUTO_INCREMENT (Row Number starting at 1)
df_gold_icd10 = df_silver_icd10.withColumn("icd10_sk", row_number().over(window_spec))

# 3. Select ONLY the columns required in Gold (Column Pruning & Reordering)
df_gold_icd10 = df_gold_icd10.select(
    "icd10_sk",
    "icd10_code",          # Business Key
    "icd10_description",   # Attribute
    "category",            # Hierarchy Attribute
    "severity",            # Attribute (Added from actual data)
    "is_preventable"       # Attribute (Added from actual data)
)

display(df_gold_icd10.limit(5))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_icd10_codes"

df_gold_icd10.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_ICD10_Codes to Gold layer.")